# Semana 5 — O Training Loop: nn.Module, DataLoader e MLP
**Fase 2: Deep Learning e o Ecossistema Hugging Face** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/applied-ai-architecture/blob/main/02-Phase-2-Deep-Learning/notebooks/semana_05_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-05-Conceitos.md](https://github.com/NetoRibeiro/applied-ai-architecture/blob/main/02-Phase-2-Deep-Learning/docs/Semana-05-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_05_gabarito.ipynb`.

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
device = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0); print(device)

## Passo 1 — Um problema que uma reta não resolve

In [ ]:
X, y = make_moons(n_samples=600, noise=0.28, random_state=0)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.5, random_state=0)
plt.scatter(*X_tr.T, c=y_tr, s=10, cmap="coolwarm"); plt.title("duas luas"); plt.show()

## Passo 2 — `Dataset` e `DataLoader`
Implemente `__len__` e `__getitem__`. Devolva tensores `float32` para `x` e `long` para o label.

In [ ]:
class Luas(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32); self.y = torch.tensor(y, dtype=torch.long)
    # [SEU CÓDIGO AQUI] implemente __len__ e __getitem__

dl_tr = DataLoader(Luas(X_tr, y_tr), batch_size=32, shuffle=True)
dl_va = DataLoader(Luas(X_va, y_va), batch_size=256)
xb, yb = next(iter(dl_tr)); print(xb.shape, yb.shape, yb.dtype)

## Passo 3 — O modelo
MLP com entrada 2, duas hidden layers de `largura` neurônios com ReLU, dropout opcional e saída com 2 logits.

In [ ]:
class MLP(nn.Module):
    def __init__(self, largura=64, dropout=0.0):
        super().__init__()
        # [SEU CÓDIGO AQUI] defina self.rede com nn.Sequential
    def forward(self, x):
        # [SEU CÓDIGO AQUI]

modelo = MLP().to(device)
print(modelo, "\nparâmetros:", sum(p.numel() for p in modelo.parameters()))

## Passo 4 — Sanidade: a loss inicial
Com 2 classes e pesos aleatórios, a entropia cruzada deve começar perto de `ln(2) ≈ 0,693`.

In [ ]:
criterio = nn.CrossEntropyLoss()
print("perda inicial:", round(criterio(modelo(xb.to(device)), yb.to(device)).item(), 3))

## Passo 5 — Uma epoch de treino e uma de avaliação

In [ ]:
def epoca_treino(modelo, dl, otimizador):
    modelo.train(); total, n = 0.0, 0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        # [SEU CÓDIGO AQUI] forward, loss, zero_grad, backward, step
        total += loss.item() * len(xb); n += len(xb)
    return total / n

@torch.no_grad()
def avaliar(modelo, dl):
    # [SEU CÓDIGO AQUI] coloque o modelo em modo de avaliação
    total, acertos, n = 0.0, 0, 0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        logits = modelo(xb)
        total += criterio(logits, yb).item() * len(xb)
        # [SEU CÓDIGO AQUI] some os acertos usando argmax nos logits
        n += len(xb)
    return total / n, acertos / n

## Passo 6 — Treinar e ler as curvas

In [ ]:
def treinar(modelo, epocas=300, lr=3e-3, weight_decay=0.0):
    otimizador = torch.optim.AdamW(modelo.parameters(), lr=lr, weight_decay=weight_decay)
    h = {"tr": [], "va": [], "acc": []}
    for _ in range(epocas):
        h["tr"].append(epoca_treino(modelo, dl_tr, otimizador))
        va, acc = avaliar(modelo, dl_va); h["va"].append(va); h["acc"].append(acc)
    return h

torch.manual_seed(0)
grande = MLP(largura=256).to(device); h1 = treinar(grande)
print(f"sem regularização  | treino {h1['tr'][-1]:.3f} | val {h1['va'][-1]:.3f} | acc {h1['acc'][-1]:.3f}")

## Passo 7 — Regularization
Treine o mesmo modelo com `dropout=0.3` e `weight_decay=0.05` e compare as curvas.

In [ ]:
torch.manual_seed(0)
# [SEU CÓDIGO AQUI] crie `regularizado` e treine guardando o histórico em h2
print(f"com regularização  | treino {h2['tr'][-1]:.3f} | val {h2['va'][-1]:.3f} | acc {h2['acc'][-1]:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for a, h, t in zip(ax, (h1, h2), ("sem regularização", "dropout + weight decay")):
    a.plot(h["tr"], label="treino"); a.plot(h["va"], label="validação"); a.set_title(t); a.set_xlabel("época"); a.legend()
plt.show()

## Passo 8 — A fronteira de decisão

In [ ]:
@torch.no_grad()
def fronteira(modelo, titulo):
    modelo.eval()
    gx, gy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
    grade = torch.tensor(np.c_[gx.ravel(), gy.ravel()], dtype=torch.float32, device=device)
    p = torch.softmax(modelo(grade), dim=1)[:, 1].reshape(gx.shape).cpu()
    plt.contourf(gx, gy, p, levels=20, cmap="coolwarm", alpha=.6); plt.scatter(*X_va.T, c=y_va, s=8, cmap="coolwarm"); plt.title(titulo); plt.show()

fronteira(grande, "sem regularização"); fronteira(regularizado, "regularizado")

## Passo 9 — Salvar e recarregar
Salve o `state_dict`, crie um modelo novo, carregue os pesos e confirme que a accuracy é a mesma.

In [ ]:
# [SEU CÓDIGO AQUI] salve em "mlp.pt", crie `copia` e carregue os pesos
assert avaliar(copia, dl_va)[1] == avaliar(regularizado, dl_va)[1]; print("pesos recarregados com sucesso")